In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

if not os.path.exists("scripts/data.py"):
    raise FileNotFoundError(
        "Run this notebook from the repository root or the scripts directory."
    )

if "scripts" not in sys.path:
    sys.path.insert(0, "scripts")

os.makedirs("results", exist_ok=True)
os.makedirs("cache", exist_ok=True)

In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from evaluate import (
    LABEL_ORDER,
    evaluate_predictions,
    build_prediction_table,
    save_evaluation_outputs,
    print_evaluation_summary,
    reorder_probabilities_by_label_order
)

In [ ]:
# Config

# "validation": train OOF -> validation evaluation
# "test": train+validation OOF -> held-out test evaluation
RUN_MODE = "test"
VALID_RUN_MODES = {"validation", "test"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"Invalid RUN_MODE: {RUN_MODE}")

# Choose any subset from "text", "progression", and "header".
COMPONENTS = ["text", "progression"]
# COMPONENTS = ["text"]
# COMPONENTS = ["progression"]
# COMPONENTS = ["header"]
# COMPONENTS = ["text", "progression", "header"]

K = 5
ENCODER_SHORT_NAME = "pubmedbert"
TEXT_FUSION_MODE = "late_prob_timeline"
MODEL_NAME = "meta_logreg"
RANDOM_STATE = 42

component_name = "_plus_".join(COMPONENTS)

if RUN_MODE == "validation":
    COMPONENT_OUTPUT_ROOT = "results/oof_components"
    LOCAL_OUTPUT_ROOT = "results/validation/local_meta"

    FIT_SPLIT_KEY = "train"
    EVAL_SPLIT_KEY = "val"
    FIT_SPLIT_NAME = "train"
    EVAL_SPLIT_NAME = "validation"
    FIT_FILENAME = "train_oof_predictions.csv"
    EVAL_FILENAME = "val_predictions.csv"

else:
    COMPONENT_OUTPUT_ROOT = "results/oof_components/test"
    LOCAL_OUTPUT_ROOT = "results/test/local_meta"

    FIT_SPLIT_KEY = "trainval"
    EVAL_SPLIT_KEY = "test"
    FIT_SPLIT_NAME = "train+validation"
    EVAL_SPLIT_NAME = "test"
    FIT_FILENAME = "trainval_oof_predictions.csv"
    EVAL_FILENAME = "test_predictions.csv"

if "text" in COMPONENTS:
    output_dir = os.path.join(
        LOCAL_OUTPUT_ROOT,
        f"{component_name}_{ENCODER_SHORT_NAME}_k{K}_"
        f"{TEXT_FUSION_MODE}_logreg",
    )
else:
    output_dir = os.path.join(
        LOCAL_OUTPUT_ROOT,
        f"{component_name}_k{K}_logreg",
    )

print("Run mode:", RUN_MODE)
print("Components:", COMPONENTS)
print("Fitting split:", FIT_SPLIT_NAME)
print("Evaluation split:", EVAL_SPLIT_NAME)
print("Component output root:", COMPONENT_OUTPUT_ROOT)
print("Local output directory:", output_dir)
print("Label order:", LABEL_ORDER)

In [ ]:
# Prediction file paths from 01_generate_oof_probabilities.ipynb

text_dir = os.path.join(
    COMPONENT_OUTPUT_ROOT,
    "text_embedding",
    f"{ENCODER_SHORT_NAME}_k{K}_{TEXT_FUSION_MODE}_logreg",
)
progression_dir = os.path.join(
    COMPONENT_OUTPUT_ROOT,
    "progression",
    f"k{K}_logreg",
)
header_dir = os.path.join(
    COMPONENT_OUTPUT_ROOT,
    "header_embedding",
    f"{ENCODER_SHORT_NAME}_k{K}_logreg",
)

prediction_paths = {
    "text": {
        "dir": text_dir,
        "fit": os.path.join(text_dir, FIT_FILENAME),
        "eval": os.path.join(text_dir, EVAL_FILENAME),
        "manifest": os.path.join(text_dir, "feature_manifest.json"),
    },
    "progression": {
        "dir": progression_dir,
        "fit": os.path.join(progression_dir, FIT_FILENAME),
        "eval": os.path.join(progression_dir, EVAL_FILENAME),
        "manifest": os.path.join(progression_dir, "feature_manifest.json"),
    },
    "header": {
        "dir": header_dir,
        "fit": os.path.join(header_dir, FIT_FILENAME),
        "eval": os.path.join(header_dir, EVAL_FILENAME),
        "manifest": os.path.join(header_dir, "feature_manifest.json"),
    },
}

for component in COMPONENTS:
    print(f"\n{component} paths:")
    for key, path in prediction_paths[component].items():
        print(f"  {key}: {path}")


In [ ]:
def check_prediction_files(prediction_paths, components):
    missing_files = []

    for component in components:
        for key in ["fit", "eval", "manifest"]:
            path = prediction_paths[component][key]
            if not os.path.exists(path):
                missing_files.append(path)

    if missing_files:
        print("Missing files:")
        for path in missing_files:
            print(path)

        raise FileNotFoundError(
            "Some required component files are missing. Run 01_generate_oof_probabilities.ipynb"
            f"with RUN_MODE='{RUN_MODE}' and the matching K/encoder settings first."
        )

    print("All required component files found.")


check_prediction_files(prediction_paths, COMPONENTS)


In [ ]:
def get_component_prefix(component):
    if component == "text":
        return "text_prob"
    elif component == "progression":
        return "prog_prob"
    elif component == "header":
        return "header_prob"
    else:
        raise ValueError("Unknown component: " + str(component))


def load_feature_cols_from_manifest(component, prediction_paths):
    manifest_path = prediction_paths[component]["manifest"]

    if not os.path.exists(manifest_path):
        raise FileNotFoundError(f"Missing feature manifest: {manifest_path}")

    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    if "feature_cols_for_meta" not in manifest:
        raise ValueError(f"Manifest missing feature_cols_for_meta: {manifest_path}")

    return manifest["feature_cols_for_meta"]


def get_meta_feature_columns(components, prediction_paths):
    feature_cols = []

    for component in components:
        component_feature_cols = load_feature_cols_from_manifest(
            component=component,
            prediction_paths=prediction_paths
        )
        feature_cols.extend(component_feature_cols)

    return feature_cols


feature_cols = get_meta_feature_columns(
    components=COMPONENTS,
    prediction_paths=prediction_paths
)

print("Meta feature columns:")
for col in feature_cols:
    print(col)

print("Number of meta features:", len(feature_cols))

In [ ]:
def load_component_predictions(path, component, prediction_paths):
    df = pd.read_csv(path)

    required_prob_cols = load_feature_cols_from_manifest(
        component=component,
        prediction_paths=prediction_paths
    )

    missing_cols = []
    for col in required_prob_cols:
        if col not in df.columns:
            missing_cols.append(col)

    if len(missing_cols) > 0:
        raise ValueError(
            f"Missing probability columns for component {component}: {missing_cols}"
        )

    if "paper_id" not in df.columns:
        if "pmcid" in df.columns:
            df["paper_id"] = df["pmcid"]
        else:
            raise ValueError(f"{path} is missing paper_id and pmcid")

    if "section_id" not in df.columns:
        raise ValueError(f"{path} is missing section_id")

    if "y_true" not in df.columns:
        raise ValueError(f"{path} is missing y_true")

    keep_cols = [
        "paper_id",
        "section_id",
        "y_true",
    ]

    optional_cols = [
        "pmcid",
        "pmid",
        "section_order",
        "section_header",
        "section_header_clean",
        "split",
        "start_progression",
        "end_progression",
        "k_requested",
        "num_sentences_available",
        "sentence_ids",
        "section_positions",
        "component",
        "feature_mode",
        "probability_source",
        "oof_fold",
    ]

    for col in optional_cols:
        if col in df.columns:
            keep_cols.append(col)

    for col in required_prob_cols:
        keep_cols.append(col)

    out = df[keep_cols].copy()

    return out

In [ ]:
def merge_component_predictions(split_name, components, prediction_paths):
    merged_df = None

    for component in components:
        path = prediction_paths[component][split_name]

        component_df = load_component_predictions(
            path=path,
            component=component,
            prediction_paths=prediction_paths
        )

        print(split_name, component, component_df.shape)

        component_feature_cols = load_feature_cols_from_manifest(
            component=component,
            prediction_paths=prediction_paths
        )

        if merged_df is None:
            merged_df = component_df.copy()
        else:
            merge_cols = ["paper_id", "section_id"]
            keep_cols = merge_cols + ["y_true"] + component_feature_cols

            component_df_small = component_df[keep_cols].copy()

            before_rows = len(merged_df)

            merged_df = merged_df.merge(
                component_df_small,
                on=["paper_id", "section_id"],
                how="inner",
                suffixes=("", f"_{component}")
            )

            after_rows = len(merged_df)

            if after_rows != before_rows:
                raise ValueError(
                    f"Row count changed after merging {component}: "
                    f"{before_rows} -> {after_rows}. Check missing component predictions."
                )

            y_true_component_col = f"y_true_{component}"
            if y_true_component_col in merged_df.columns:
                mismatch_count = (
                    merged_df["y_true"].astype(str)
                    != merged_df[y_true_component_col].astype(str)
                ).sum()

                if mismatch_count > 0:
                    raise ValueError(
                        f"y_true mismatch after merging {component}: {mismatch_count} rows."
                    )

                merged_df = merged_df.drop(columns=[y_true_component_col])

        # if merged_df is None:
        #     merged_df = component_df.copy()
        # else:
        #     merge_cols = ["paper_id", "section_id"]
        #     keep_cols = merge_cols + component_feature_cols

        #     component_df_small = component_df[keep_cols].copy()

        #     merged_df = merged_df.merge(
        #         component_df_small,
        #         on=["paper_id", "section_id"],
        #         how="inner"
        #     )

    return merged_df

In [ ]:
# Load and merge fitting/evaluation component features.
fit_meta_df = merge_component_predictions(
    split_name="fit",
    components=COMPONENTS,
    prediction_paths=prediction_paths,
)

eval_meta_df = merge_component_predictions(
    split_name="eval",
    components=COMPONENTS,
    prediction_paths=prediction_paths,
)

print()
print(f"Merged {FIT_SPLIT_NAME} shape:", fit_meta_df.shape)
print(f"Merged {EVAL_SPLIT_NAME} shape:", eval_meta_df.shape)

print()
print("Fitting-data columns:")
print(fit_meta_df.columns.tolist())


In [ ]:
# Sanity checks.
def check_meta_dataframe(df, feature_cols, split_name):
    print("=" * 60)
    print("Checking", split_name)
    print("=" * 60)

    print("Rows:", len(df))
    print("Unique paper_id:", df["paper_id"].nunique())
    print("Unique section_id:", df["section_id"].nunique())

    duplicate_count = df.duplicated(subset=["paper_id", "section_id"]).sum()
    print("Duplicate paper_id + section_id rows:", duplicate_count)
    if duplicate_count > 0:
        raise ValueError(f"Duplicate rows found in {split_name} meta dataframe.")

    missing_features = [col for col in feature_cols if col not in df.columns]
    if missing_features:
        raise ValueError(f"Missing feature columns in {split_name}: {missing_features}")

    missing_values = int(df[feature_cols].isna().sum().sum())
    print("Missing feature values:", missing_values)
    if missing_values > 0:
        raise ValueError(f"Missing feature values found in {split_name}.")

    print()
    print("Label counts:")
    print(df["y_true"].value_counts().reindex(LABEL_ORDER))

    print()
    print("Feature-sum checks by component:")
    for component in COMPONENTS:
        component_feature_cols = load_feature_cols_from_manifest(
            component=component,
            prediction_paths=prediction_paths,
        )
        row_sums = df[component_feature_cols].sum(axis=1)
        print(component)
        print(row_sums.describe())


check_meta_dataframe(fit_meta_df, feature_cols, FIT_SPLIT_NAME)
check_meta_dataframe(eval_meta_df, feature_cols, EVAL_SPLIT_NAME)


In [ ]:
# Build fitting/evaluation matrices.
X_fit = fit_meta_df[feature_cols].values
y_fit = fit_meta_df["y_true"].values

X_eval = eval_meta_df[feature_cols].values
y_eval = eval_meta_df["y_true"].values

print("X_fit shape:", X_fit.shape)
print("X_eval shape:", X_eval.shape)
print("y_fit shape:", y_fit.shape)
print("y_eval shape:", y_eval.shape)


In [ ]:
# Train the local classifier.
meta_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "logreg",
            LogisticRegression(
                max_iter=1000,
                multi_class="multinomial",
                solver="lbfgs",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

meta_model.fit(X_fit, y_fit)
print("Local classifier trained on", FIT_SPLIT_NAME)


In [ ]:
# Predict on the evaluation split.
y_pred = meta_model.predict(X_eval)
y_prob_raw = meta_model.predict_proba(X_eval)

logreg_model = meta_model.named_steps["logreg"]
y_prob = reorder_probabilities_by_label_order(
    model_classes=logreg_model.classes_,
    y_prob=y_prob_raw,
    label_order=LABEL_ORDER,
)

print("Evaluation split:", EVAL_SPLIT_NAME)
print("Model class order:", list(logreg_model.classes_))
print("Fixed label order:", LABEL_ORDER)
print("Probability shape:", y_prob.shape)


In [ ]:
# Evaluate
metrics, report_df, confusion_df = evaluate_predictions(
    y_true=y_eval,
    y_pred=y_pred,
    y_prob=y_prob,
    label_order=LABEL_ORDER,
)

print_evaluation_summary(metrics, confusion_df)

print()
print("Classification report:")
display(report_df)

print()
print("Confusion matrix:")
display(confusion_df)


In [ ]:
# Build prediction table
id_cols = []

candidate_id_cols = [
    "paper_id",
    "pmcid",
    "pmid",
    "section_id",
    "section_order",
    "section_header",
    "split",
    "start_progression",
    "end_progression",
    "k_requested",
    "num_sentences_available",
    "sentence_ids",
    "section_positions",
]

for col in candidate_id_cols:
    if col in eval_meta_df.columns:
        id_cols.append(col)

eval_ids_df = eval_meta_df[id_cols].copy()
eval_ids_df["k"] = K
eval_ids_df["model_stage"] = "local_meta_classifier"
eval_ids_df["model_name"] = MODEL_NAME
eval_ids_df["components"] = component_name

predictions_df = build_prediction_table(
    df_ids=eval_ids_df,
    y_true=y_eval,
    y_pred=y_pred,
    y_prob=y_prob,
    label_order=LABEL_ORDER,
    prob_prefix="meta_prob",
)

# Preserve component inputs for later error and coefficient analyses.
for col in feature_cols:
    predictions_df[col] = eval_meta_df[col].values

display(predictions_df.head())


In [ ]:
# save
config = {
    "run_mode": RUN_MODE,
    "model_stage": "local_meta_classifier",
    "model": "logistic_regression",
    "components": COMPONENTS,
    "component_name": component_name,
    "encoder_short_name": ENCODER_SHORT_NAME if "text" in COMPONENTS else None,
    "text_fusion_mode": TEXT_FUSION_MODE if "text" in COMPONENTS else None,
    "k": K,
    "feature_cols": feature_cols,
    "num_features": len(feature_cols),
    "fit_split": FIT_SPLIT_NAME,
    "split_evaluated": EVAL_SPLIT_NAME,
    "label_order": LABEL_ORDER,
    "uses_test_split": RUN_MODE == "test",
    "fit_rows": int(len(fit_meta_df)),
    "evaluation_rows": int(len(eval_meta_df)),
    "important_note": (
        "The local classifier is trained on grouped out-of-fold component features "
        f"from {FIT_SPLIT_NAME}. Evaluation component features are produced by component "
        f"models fitted on the complete {FIT_SPLIT_NAME} split and applied to {EVAL_SPLIT_NAME}."
    ),
}

save_evaluation_outputs(
    output_dir=output_dir,
    metrics=metrics,
    report_df=report_df,
    confusion_df=confusion_df,
    predictions_df=predictions_df,
    config=config,
)

print("Saved outputs to:", output_dir)
print(os.listdir(output_dir))


In [ ]:
# Sanity check on saved local probabilities.
print("Prediction rows:", len(predictions_df))
print(f"{EVAL_SPLIT_NAME} rows:", len(eval_meta_df))

meta_prob_cols = [
    col for col in predictions_df.columns
    if col.startswith("meta_prob_")
]

print()
print("Local probability columns:")
print(meta_prob_cols)

print()
print("Local probability row sums:")
print(predictions_df[meta_prob_cols].sum(axis=1).describe())
